# ДЗ до теми 5 — Rain in Australia

Мета роботи — підготувати часові, числові й категоріальні ознаки та побудувати Logistic Regression для прогнозу `RainTomorrow`. Дані беруться з відкритого репозиторію матеріалів курсу.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
DATA_URL = 'https://raw.githubusercontent.com/goitacademy/MACHINE-LEARNING-NEO/main/datasets/mod_03_topic_05_weather_data.csv.gz'
data = pd.read_csv(DATA_URL)
print(f'Початковий розмір: {data.shape}')
display(data.head())

Початковий розмір: (145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [2]:
# Ознаки з великою часткою пропусків не використовуємо.
missing_share = data.isna().mean().sort_values(ascending=False)
data = data.loc[:, missing_share[missing_share < 0.35].index].copy()
data = data.dropna(subset=['RainTomorrow']).copy()

date = pd.to_datetime(data.pop('Date'))
y = data.pop('RainTomorrow')
X_num = data.select_dtypes(include=np.number).copy()
X_cat = data.select_dtypes(include='object').copy()

# Рік — числова ознака, а місяць — категоріальна.
X_num['Year'] = date.dt.year
X_cat['Month'] = date.dt.month.astype(str)
last_year = X_num['Year'].max()
train_mask = X_num['Year'] < last_year
test_mask = X_num['Year'] == last_year
print(f'Останній рік: {last_year}; train: {train_mask.sum()}; test: {test_mask.sum()}')
print('Числові ознаки:', X_num.columns.tolist())
print('Категоріальні ознаки:', X_cat.columns.tolist())

Останній рік: 2017; train: 133727; test: 8466
Числові ознаки: ['Pressure9am', 'Pressure3pm', 'WindGustSpeed', 'Humidity3pm', 'Temp3pm', 'Rainfall', 'WindSpeed3pm', 'Humidity9am', 'Temp9am', 'WindSpeed9am', 'MinTemp', 'MaxTemp', 'Year']
Категоріальні ознаки: ['WindDir9am', 'WindGustDir', 'WindDir3pm', 'RainToday', 'Location', 'Month']


In [3]:
X_train_num, X_test_num = X_num.loc[train_mask], X_num.loc[test_mask]
X_train_cat, X_test_cat = X_cat.loc[train_mask], X_cat.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
scaler = StandardScaler()
encoder = OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False)

X_train_num_ready = scaler.fit_transform(num_imputer.fit_transform(X_train_num))
X_test_num_ready = scaler.transform(num_imputer.transform(X_test_num))
X_train_cat_ready = encoder.fit_transform(cat_imputer.fit_transform(X_train_cat))
X_test_cat_ready = encoder.transform(cat_imputer.transform(X_test_cat))

X_train = np.hstack([X_train_num_ready, X_train_cat_ready])
X_test = np.hstack([X_test_num_ready, X_test_cat_ready])
print('Підготовлені матриці:', X_train.shape, X_test.shape)

Підготовлені матриці: (133727, 123) (8466, 123)


In [4]:
model = LogisticRegression(
    solver='liblinear', class_weight='balanced', max_iter=1000, random_state=42
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(classification_report(y_test, prediction, digits=3))

              precision    recall  f1-score   support

          No      0.917     0.819     0.865      6703
         Yes      0.511     0.720     0.598      1763

    accuracy                          0.798      8466
   macro avg      0.714     0.769     0.732      8466
weighted avg      0.833     0.798     0.810      8466



## Висновок

Тестова вибірка сформована з останнього року спостережень, тому оцінка імітує прогноз для майбутніх даних. Пропуски й перетворення були навчені лише на train-вибірці, що не допускає витоку даних.

Порівняно з базовою моделлю з матеріалів теми, яка на випадковій test-вибірці має accuracy близько 0.79, macro F1 близько 0.73 і recall класу `Yes` 0.76, ця модель на часовій test-вибірці має accuracy 0.798, macro F1 0.732 і recall класу `Yes` 0.720. Метрики близькі, але часовий поділ є суворішим: модель перевіряється на даних наступного року, яких не бачила під час навчання.